In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
session = requests.Session()
retries = Retry(total=5, backoff_factor=1, status_forcelist=[500, 502, 503, 504])
session.mount('http://', HTTPAdapter(max_retries=retries))
session.mount('https://', HTTPAdapter(max_retries=retries))
available_timezones_url = "https://www.timeapi.io/api/TimeZone/AvailableTimeZones"
try:
    response = session.get(available_timezones_url, timeout=10)
    response.raise_for_status()
    timezones = response.json()
except requests.exceptions.RequestException as e:
    print(f"Error fetching timezones list: {e}")
    timezones = []

search = input("Enter location (usa/london/india): ").lower()

result = []
if search == "usa":

    result = [tz for tz in timezones if tz.startswith("America/")]
elif search == "london":

    result = [tz for tz in timezones if "Europe/London" in tz]
elif search == "india":

    result = [tz for tz in timezones if "Asia/Kolkata" in tz]
else:
    print("No specific filter applied for the entered location. Showing all found.")
    result = timezones

if result:

    if search not in ["usa", "london", "india"] and len(result) > 10:
        print(f"Displaying first 10 of {len(result)} timezones. Please be more specific with your search.")
        result = result[:10]


    specific_timezone_data_url = "https://www.timeapi.io/api/TimeZone/zone"

    for tz_name in result:
        try:

            data_response = session.get(specific_timezone_data_url, params={'timeZone': tz_name}, timeout=10)
            data_response.raise_for_status()
            data = data_response.json()
            if data:
                current_time = data.get('currentLocalTime')
                print(f"\nTimezone: {tz_name}")
                print(f"Current Time: {current_time if current_time else 'N/A'}")
            else:
                print(f"\nCould not retrieve data for timezone: {tz_name}")
        except requests.exceptions.RequestException as e:
            print(f"\nError fetching data for {tz_name}: {e}")
else:
    print("No matching timezone found.")